In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print("Q1 Answer:", len(documents))

Q1 Answer: 72


In [2]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

print("\nQ2 Results:")

for r in results:
    print(r["filename"])

print("\nQ2 Answer:", results[0]["filename"])


Q2 Results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md

Q2 Answer: 01-agentic-rag/lessons/14-agentic-loop.md


In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    base_url="https://ai-gateway.vercel.sh/v1",
    
    api_key=os.getenv("OPENAI_API_KEY"), 
)

In [4]:
def search(query):
    return index.search(
        query=query,
        num_results=5
    )

In [5]:
def build_context(results):

    context = ""

    for doc in results:
        context += f"""
FILE: {doc['filename']}

{doc['content']}

=========================
"""

    return context

In [6]:
def rag(query):

    results = search(query)

    context = build_context(results)

    prompt = f"""
You are a teaching assistant.

Answer the question based on the context.

QUESTION:
{query}

CONTEXT:
{context}
"""

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    answer = response.output_text

    tokens = response.usage.input_tokens

    return answer, tokens

In [14]:
answer, tokens = rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("\nQ3 Answer:")
print(answer)

print("\nQ3 Input Tokens:")
print(tokens)


Q3 Answer:
The agentic loop keeps calling the model inside a `while True` loop.

Each turn it:

1. sends the current `messages` history to the model,
2. checks the response for any `function_call` items,
3. runs those tools and appends the results to `messages`,
4. sets `has_function_calls = True` if any tool was called.

At the end of the iteration:

```python
if has_function_calls == False:
    break
```

So it stops only when the model returns a response with no function calls, meaning it has given its final answer.

Q3 Input Tokens:
7104


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print("\nQ4 Number of Chunks:")
print(len(chunks))


Q4 Number of Chunks:
295


In [8]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [9]:
def chunk_search(query):
    return chunk_index.search(
        query=query,
        num_results=5
    )

In [10]:
def chunk_rag(query):

    results = chunk_search(query)

    context = ""

    for doc in results:
        context += f"""
FILE: {doc['filename']}
START: {doc['start']}

{doc['content']}

======================
"""

    prompt = f"""
You are a teaching assistant.

Answer the question using the context.

QUESTION:
{query}

CONTEXT:
{context}
"""

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return (
        response.output_text,
        response.usage.input_tokens
    )

In [19]:
_, original_tokens = rag(
    "How does the agentic loop keep calling the model until it stops?"
)

_, chunk_tokens = chunk_rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("\nOriginal Tokens:", original_tokens)
print("Chunk Tokens:", chunk_tokens)

print(
    "Reduction Ratio:",
    round(original_tokens / chunk_tokens, 2)
)


Original Tokens: 7104
Chunk Tokens: 2315
Reduction Ratio: 3.07


In [11]:
search_calls = 0

In [12]:
def search(query: str) -> str:
    """
    Search the course lessons.
    """

    global search_calls

    search_calls += 1

    results = chunk_index.search(
        query=query,
        num_results=5
    )

    context = ""

    for r in results:
        context += f"""
FILE: {r['filename']}

{r['content']}

====================
"""

    return context

In [13]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

In [14]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
instructions = """
You're a course teaching assistant.

Answer the student's question using the search tool.

Make multiple searches with different keywords before answering.
"""

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv('.env')
openai_client = OpenAI(
    base_url="https://ai-gateway.vercel.sh/v1",
    
    api_key=os.getenv("OPENAI_API_KEY"), 
)

In [18]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import (
    OpenAIResponsesRunner,
    DisplayingRunnerCallback
)

chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(
    chat_interface
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="gpt-5.4-mini",
        client=openai_client
    )
)

In [19]:
result = runner.loop(
    prompt="""
How does the agentic loop work,
and how is it different from plain RAG?
""",
    callback=callback
)

-> Response received


-> Response received


In [20]:
print("Search calls:", search_calls)

Search calls: 3
